# Learn Demeter by running small experiments

Follow [the learning path](../docs/LEARNING_PATH.md) after installation.
All modeled health effects below are **validation-only**, not findings.
Run all cells in order. No network, server or calibration is required.
From the repo root: `uv run python scripts/run_learning_notebook.py`.
The script executes the same cells without a notebook UI. In an existing Jupyter
editor, select the project's `.venv` Python kernel; start from the repo or
`notebooks` directory. Executing this trusted notebook executes Python code.

In [ ]:
import json
import math
from pathlib import Path
import sys
import yaml
from demeter.schema import EvidenceRegistry, Scenario
from demeter.model import simulate
from demeter.examples.stock_flow import run as run_toy
from demeter.analysis.experiments import compare, with_values

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "evidence/parameters.yaml").is_file(), "Start in the Demeter checkout"
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
destination = Path(globals().get("TUTORIAL_DESTINATION", root / "outputs/tutorial"))
destination.mkdir(parents=True, exist_ok=True)

registry = EvidenceRegistry.from_yaml(root / "evidence/parameters.yaml")
def save(name, value):
    (destination / name).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n",
                                    encoding="utf-8", newline="\n")

## Start with two containers

Predict the first transfer before running. Both stocks use abstract tokens;
one step is a tick, not a year. Response closes half the gap to the target 0.25,
so the first fraction is 0.125 and 12.5 of the initial 100 tokens move.
The next transfer uses the remaining stock, not the initial stock. Values and
illustrative ranges are in the registry. Nothing here is a clinical rate.

In [ ]:
toy = run_toy(registry)
assert toy["rows"][1]["transfer"] == 12.5
assert toy["rows"][2]["remaining"] == 71.09375
assert all(math.isclose(r["remaining"] + r["received"], 100) for r in toy["rows"])
save("toy.json", toy)
print(json.dumps(toy["rows"], indent=2))

## Feedback, delay and uncertainty

At a constant effective fraction, shrinking remaining stock reduces later flow:
balancing feedback. With gradual adjustment, early flow can first rise as the
fraction grows. A response fraction of one removes the delay; zero target removes
all flow. Compare endpoints of the registered target range using copied evidence.
These are sensitivity cases, not sampled confidence bounds or calibration.

In [ ]:
instant = run_toy(with_values(registry, {"toy_response_fraction": 1}))
assert instant["rows"][1]["transfer"] == 25
assert all(a["transfer"] > b["transfer"] for a, b in zip(instant["rows"][1:-1], instant["rows"][2:], strict=True))
zero = run_toy(with_values(registry, {"toy_target_fraction": 0}))
assert zero["rows"][-1]["remaining"] == 100
u = registry.parameters["toy_target_fraction"].uncertainty
low = run_toy(with_values(registry, {"toy_target_fraction": u.low}))
high = run_toy(with_values(registry, {"toy_target_fraction": u.high}))
assert low["rows"][-1]["received"] < toy["rows"][-1]["received"] < high["rows"][-1]["received"]
save("toy-alternatives.json", {"instant": instant, "zero": zero, "low": low, "high": high})

## 2. Canonical baseline

The health engine has different semantics: annual hazards, explicit deaths,
aging and life tables. Token fractions must never be substituted for its hazards.
Conservation means ending living population plus cumulative deaths equals starting
population, because these scenarios have no births or migration.

In [ ]:
baseline = Scenario.from_yaml(root / "scenarios/baseline.yaml")
intervention = Scenario.from_yaml(root / "scenarios/reduce_upf_30.yaml")
base = simulate(registry, baseline)
assert base.validation_only
assert len(base.annual) == baseline.years + 1
assert len(base.cohorts) == 101
assert math.isclose(base.ending_population + base.cumulative_deaths, base.starting_population)
save("baseline.json", base.to_dict())
print({"annual_boundaries": len(base.annual), "age_cells": len(base.cohorts), "validation_only": base.validation_only})

## 3. Inspect stocks, flows and the causal graph

Generate the existing offline report once, including uncertainty, sensitivity
and historical diagnostics used later. Open `outputs/tutorial/report/index.html`.
Inspect **Living stocks**, **Annual flows** and
**Implemented dependency graph**. Dashed synthetic links are assumptions.
Graph arrows describe implemented dependencies; they do not prove causation.

In [ ]:
from demeter.analysis.observability import observe
from demeter.analysis.visualization import render_report, build_figures

payload = observe(registry, intervention, draws=4, samples=8, seed=42)
report = render_report(payload, destination / "report")
assert Path(report["report"]).is_file()
figures = build_figures(payload)
figures["dependencies"]

## 4–5. Change one assumption and compare results and uncertainty

The supplied reduction changes UPF from 1 to 0.7, a relative exposure, not an
absolute dietary share. Read absolute deltas and their units before percentages.
Period life expectancy freezes a mortality schedule; it is not a forecast of
individual lifespan. Monte Carlo uses paired baseline/intervention draws.
Four draws and eight Sobol base samples are smoke tests, not stable estimates.

In [ ]:
comparison = compare(registry, baseline, intervention)
assert comparison["validation_only"]
save("comparison.json", comparison)
save("uncertainty.json", payload["uncertainty"])
print(comparison["outcomes"]["life_expectancy"])
print(payload["uncertainty"]["paired_deltas"]["life_expectancy"])

## 6. Follow a weak link back to its evidence

`beta_upf_progression` is synthetic, grade E. Its range is a development
assumption. The clinical candidate below is benchmark-only and has unresolved
transport blockers; a published estimate is not automatically a usable causal
coefficient. Status describes provenance, grade describes strength separately.

In [ ]:
effect = registry.parameters["beta_upf_progression"]
assert effect.status == "synthetic" and effect.evidence_grade == "E"
candidate = registry.parameters["aric_hba1c_prediabetes_diabetes_incidence"]
assert candidate.model_role == "benchmark_only" and candidate.applicability.blockers
save("evidence-example.json", {"active_assumption": effect.model_dump(mode="json"),
                              "candidate": candidate.model_dump(mode="json")})
print(json.dumps(candidate.applicability.model_dump(), indent=2))

## 7–8. Sensitivity and historical validation

Sensitivity asks which uncertain assumptions drive output variation; it does not
upgrade their evidence. Read the saved Sobol diagnostics without treating this
small run as a ranking. Historical backtests compare predictions against held-out
observations. The short mortality persistence example carries a training schedule
forward unchanged; it neither fits coefficients nor validates dietary effects.
The report also contains rolling historical folds, residuals and limitations.

In [ ]:
from demeter.analysis.validation import mortality_backtest

save("sensitivity.json", payload["sensitivity"])
save("historical.json", payload["historical"])
backtest = mortality_backtest(2022, 2023)
assert not backtest["holdout_used_in_prediction"]
assert not backtest["scientific_validation_of_diet"]
save("mortality-backtest.json", backtest)
print(backtest)

## 9. Create and reload a scenario

Make a control from the intervention and remove only the UPF change. Expected
absolute differences from baseline are exactly zero. The YAML is saved and parsed
again, so the exercise checks the same persisted input a CLI run would load.

In [ ]:
control_data = intervention.model_dump(mode="json")
control_data.update(name="learning_control", description="Null exposure change for software validation")
control_data["exposures"]["upf"] = 1.0
control_path = destination / "control.yaml"
control_path.write_text(yaml.safe_dump(control_data, sort_keys=False), encoding="utf-8", newline="\n")
control = Scenario.from_yaml(control_path)
null_comparison = compare(registry, baseline, control)
assert all(row["absolute_delta"] == 0 for row in null_comparison["outcomes"].values())
save("control-comparison.json", null_comparison)

## 10. Add a custom module

Read and copy `examples/tutorial_module.py` as described in the guide. It uses the
public contract to implement the existing structural null: hazards ignore dietary
modifiers. The class must live in an inspectable Python file for source provenance.
Do not define it only in a notebook cell. The engine retains population accounting.
The equality below tests a null hypothesis implementation, not clinical truth.

In [ ]:
from examples.tutorial_module import LearningNull

custom = simulate(registry, intervention, transition_module=LearningNull())
assert custom.cohorts == base.cohorts
assert custom.validation_only
assert math.isclose(custom.ending_population + custom.cumulative_deaths, custom.starting_population)
save("module-null.json", custom.to_dict())
summary = {"validation_only": True, "registry_sha256": registry.content_hash,
           "toy_first_transfer": toy["rows"][1]["transfer"],
           "toy_second_remaining": toy["rows"][2]["remaining"],
           "baseline_age_cells": len(base.cohorts), "control_deltas_zero": True,
           "module_null_matches_baseline": True, "calibration_performed": False}
save("checks.json", summary)
print(json.dumps(summary, indent=2))

## Debrief

Record your prediction, what changed, and which observation could reject the
mechanism. Calibration means choosing/fitting parameters against specified
observations; validation tests performance on appropriate independent data.
Passing arithmetic tests or matching one life table does not validate the whole
health pathway. Calibration remains a separate reviewed exercise. Keep the Git
commit, locked environment, scenarios, module source, registry and seed with any
results. No diet, healthcare-cost or investment conclusion follows from this run.